# TypeOracle Colab Validation

**Purpose:** Validate the symbolic TypeOracle (DCA-Trie) on WebQSP using a
lightweight model (Qwen2.5-1.5B) before running the full experiment with
Llama-3.1-8B.

**What this notebook does:**
1. Evaluates TypeOracle SIR/FNR metrics (CPU-only, no GPU needed)
2. Loads Qwen2.5-1.5B-Instruct for constrained decoding
3. Runs graph-constrained decoding with TypeOracle-filtered paths
4. Runs unfiltered baseline for comparison
5. Compares Hit@1 and path reduction
6. Saves all results to Google Drive

In [ ]:
#@title 1. Setup & Environment

# Mount Google Drive.
import os
import sys
import time
import json

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# GPU Detection.
import torch

# GPU arch mapping: name substring -> SM version for flash-attn.
GPU_ARCH_MAP = {
    "A100": "80",
    "A100-SXM": "80",
    "A100-PCIE": "80",
    "L4": "89",
    "T4": "75",
    "V100": "70",
    "H100": "90",
    "H200": "90",
    "RTX 4090": "89",
    "RTX 3090": "86",
}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    gpu_arch = None
    for key, arch in GPU_ARCH_MAP.items():
        if key in gpu_name:
            gpu_arch = arch
            break
    has_a100 = "A100" in gpu_name
    print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  SM arch: {gpu_arch}")
else:
    gpu_name = "None"
    gpu_mem = 0
    gpu_arch = None
    has_a100 = False
    print("WARNING: No GPU detected. Inference will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

# Clone repo (skip if already cloned).
REPO_DIR = "/content/graph-constrained-reasoning"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, '.')

# Install core dependencies (pinned versions).
DEPS = [
    "transformers==4.44.2",
    "accelerate>=0.30.1",
    "datasets>=2.19.2",
    "marisa-trie>=1.2.0",
    "networkx>=3.0",
    "scikit-learn>=1.5.0",
    "tiktoken>=0.7.0",
    "sentencepiece>=0.2.0",
    "protobuf>=5.27.1",
]

print("Installing core dependencies...")
!pip install -q {' '.join(DEPS)}
print("Done.")

# Install flash-attn (optional).
# Not required -- sdpa works on all GPUs. But on A100,
# flash_attention_2 is ~20% faster for beam search.
#
# Strategy:
#   1. Try pre-compiled wheel from GitHub releases (instant)
#   2. Source install with ninja + arch targeting (~10-15 min)
#   3. Fall back to sdpa (no flash-attn)

INSTALL_FLASH_ATTN = False  #@param {type:"boolean"}

flash_attn_installed = False

if INSTALL_FLASH_ATTN and torch.cuda.is_available():
    # Detect Python + CUDA + PyTorch versions for wheel matching.
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    cuda_ver = torch.version.cuda.replace(".", "")
    torch_ver = torch.__version__.split("+")[0]
    torch_major_minor = ".".join(torch_ver.split(".")[:2])

    print(f"Python: {py_ver}  CUDA: {cuda_ver}  PyTorch: {torch_ver}")

    # Step 1: Try pre-compiled wheel.
    wheel_url = (
        f"https://github.com/Dao-AILab/flash-attention/releases/"
        f"download/v2.7.3/flash_attn-2.7.3+cu{cuda_ver}"
        f"torch{torch_major_minor}cxx11abiFALSE-cp{py_ver}"
        f"-cp{py_ver}-linux_x86_64.whl"
    )
    print("Trying pre-compiled wheel...")
    ret = os.system(f"pip install -q '{wheel_url}' 2>/dev/null")
    if ret == 0:
        flash_attn_installed = True
        print("flash-attn installed from pre-compiled wheel.")
    else:
        print("Pre-compiled wheel not available for this combination.")

    # Step 2: Source install with ninja + arch targeting.
    if not flash_attn_installed:
        print("\nFalling back to source install with ninja...")
        !pip install -q ninja

        # Set target architecture to avoid compiling for all GPU types.
        if gpu_arch:
            os.environ["FLASH_ATTN_CUDA_ARCHS"] = gpu_arch
            print(f"Targeting SM {gpu_arch} only (your GPU: {gpu_name})")

        # Ninja parallelizes compilation across all CPU cores.
        n_cores = os.cpu_count() or 1
        print(f"Compiling with ninja ({n_cores} cores), ~10-15 min...")

        t0 = time.time()
        !pip install flash-attn --no-build-isolation
        elapsed = time.time() - t0

        try:
            import flash_attn
            flash_attn_installed = True
            print(
                f"flash-attn installed in {elapsed:.0f}s "
                f"(v{flash_attn.__version__})"
            )
        except ImportError:
            print(
                f"flash-attn install failed after {elapsed:.0f}s. "
                f"Using sdpa instead."
            )

if flash_attn_installed:
    print("flash_attention_2 available.")
else:
    print("Using sdpa attention (built into transformers, no install needed).")

# Verify key imports.
from src.llms import get_registed_model
from src.trie import MarisaTrie
from src.graph_constrained_decoding import GraphConstrainedDecoding
import src.utils as utils
import networkx as nx
from datasets import load_dataset

print("\nAll imports verified.")
print(f"Working directory: {os.getcwd()}")

---
## 2. Configuration

All experiment parameters in one place. Adjust these before running.

In [ ]:
#@title 2. Configuration

# Model.
MODEL_PATH = "Qwen/Qwen2.5-1.5B-Instruct"  #@param {type:"string"}

# Dataset.
DATA_PATH = "rmanluo"   #@param {type:"string"}
DATASET = "RoG-webqsp"  #@param {type:"string"}
SPLIT = "test"          #@param {type:"string"}

# Decoding.
INDEX_LEN = 2            #@param {type:"integer"}
K = 5                    #@param {type:"integer"}
GEN_MODE = "group-beam"  #@param ["greedy", "group-beam", "beam"]
PROMPT_MODE = "zero-shot"  #@param ["zero-shot", "few-shot"]
MAX_NEW_TOKENS = 256     #@param {type:"integer"}

# Sampling.
MAX_SAMPLES = 100  #@param {type:"integer"}

# Output (Google Drive).
DRIVE_BASE = "/content/drive/MyDrive/GCR_Results"  #@param {type:"string"}
FORCE_RERUN = False  #@param {type:"boolean"}

# Derived settings (do not edit).
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(DRIVE_BASE, TIMESTAMP)
ATTN_IMPL = (
    "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Model:          {MODEL_PATH}")
print(f"Dataset:        {DATA_PATH}/{DATASET} ({SPLIT})")
print(f"Max samples:    {MAX_SAMPLES}")
print(f"Beam width:     {K}")
print(f"Index length:   {INDEX_LEN} hops")
print(f"Generation:     {GEN_MODE}")
print(f"Attention:      {ATTN_IMPL}")
print(f"GPU:            {gpu_name}")
print(f"Output:         {OUTPUT_DIR}")
print("=" * 60)

# Save config.
config = {
    "model_path": MODEL_PATH,
    "data_path": DATA_PATH,
    "dataset": DATASET,
    "split": SPLIT,
    "index_len": INDEX_LEN,
    "k": K,
    "gen_mode": GEN_MODE,
    "prompt_mode": PROMPT_MODE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_samples": MAX_SAMPLES,
    "attn_impl": ATTN_IMPL,
    "gpu": gpu_name,
    "timestamp": TIMESTAMP,
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {OUTPUT_DIR}/config.json")

---
## 3. Load Dataset

In [ ]:
#@title 3. Load Dataset

dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
print(f"Full test set: {len(dataset)} questions")

if MAX_SAMPLES is not None and MAX_SAMPLES < len(dataset):
    dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Subsampled to {len(dataset)} questions")

# Preview first 3 questions.
print("\n--- Preview ---")
for i in range(min(3, len(dataset))):
    d = dataset[i]
    g = utils.build_graph(d["graph"])
    n_edges = g.number_of_edges()
    n_nodes = g.number_of_nodes()
    print(f"\n[{i}] Q: {d['question']}")
    print(f"    A: {d['answer']}")
    print(f"    Entities: {d.get('q_entity', [])}")
    print(f"    Graph: {n_nodes} nodes, {n_edges} edges")

---
## 4. TypeOracle SIR/FNR Evaluation (CPU-only)

This phase requires **no GPU**. It evaluates the symbolic TypeOracle gates:
- **SIR** (Semantic Irrelevance Ratio): fraction of paths pruned
- **SIR_type**: paths blocked by the type gate (terminal entity type mismatch)
- **SIR_traj**: paths blocked by the range gate (relation range mismatch)
- **FNR** (False Negative Rate): fraction of gold paths incorrectly pruned

In [ ]:
#@title 4. TypeOracle SIR/FNR Evaluation

from approach3_symbolic.type_oracle import TypeOracle

print("=" * 60)
print(f"Phase 1: SIR/FNR Evaluation -- {len(dataset)} samples (CPU)")
print("=" * 60)

total_before = 0
total_after = 0
total_range_blocked = 0
total_type_blocked = 0
n_range_fn = 0
n_type_fn = 0
total_gold = 0
skipped = 0

t0 = time.time()

for i, d in enumerate(dataset):
    try:
        oracle = TypeOracle.from_graph(d["graph"])
        ans_types = oracle.infer_answer_types(d["question"])
        g = utils.build_graph(d["graph"], undirected=False)
        entities = d.get("q_entity", [])

        # SIR: enumerate all paths, apply gates.
        if entities:
            paths_list = utils.dfs(g, entities, INDEX_LEN)
            kept = []
            for p in paths_list:
                admit = True
                for _, rel, tail in p:
                    if not oracle.range_gate(rel, tail):
                        total_range_blocked += 1
                        admit = False
                        break
                if admit:
                    terminal = p[-1][2]
                    if not oracle.type_gate(
                        terminal, ans_types, len(p), INDEX_LEN
                    ):
                        total_type_blocked += 1
                        admit = False
                if admit:
                    kept.append(p)
            total_before += len(paths_list)
            total_after += len(kept)
        else:
            skipped += 1

        # FNR: check gold paths.
        truth_paths = utils.get_truth_paths(d["q_entity"], d["a_entity"], g)
        for p in truth_paths:
            if not p:
                continue
            total_gold += 1
            if any(
                not oracle.range_gate(rel, tail) for _, rel, tail in p
            ):
                n_range_fn += 1
            if not oracle.type_gate(
                p[-1][2], ans_types, len(p), INDEX_LEN
            ):
                n_type_fn += 1

    except Exception as e:
        print(f"  Skipping sample {i}: {e}")
        skipped += 1

    if (i + 1) % 25 == 0:
        elapsed = time.time() - t0
        print(f"  processed {i + 1}/{len(dataset)} ({elapsed:.1f}s)")

elapsed = time.time() - t0
pruned = total_before - total_after
sir = pruned / max(1, total_before)

sir_metrics = {
    "samples": len(dataset),
    "skipped": skipped,
    "total_paths_raw": total_before,
    "total_paths_filtered": total_after,
    "pruned": pruned,
    "sir": round(sir, 4),
    "sir_type": round(total_type_blocked / max(1, total_before), 4),
    "sir_traj": round(total_range_blocked / max(1, total_before), 4),
    "n_range_blocked": total_range_blocked,
    "n_type_blocked": total_type_blocked,
    "gold_paths": total_gold,
    "fnr_type": round(n_type_fn / max(1, total_gold), 4),
    "fnr_range": round(n_range_fn / max(1, total_gold), 4),
    "elapsed_s": round(elapsed, 1),
}

print(f"\n{'=' * 60}")
print("RESULTS -- SIR/FNR")
print(f"{'=' * 60}")
print(f"Samples:               {sir_metrics['samples']}")
print(f"Skipped:               {sir_metrics['skipped']}")
print(f"Total paths (raw):     {sir_metrics['total_paths_raw']}")
print(f"Total paths (filtered):{sir_metrics['total_paths_filtered']}")
print(f"Pruned:                {sir_metrics['pruned']}")
print(f"SIR (overall):         {sir_metrics['sir']}")
print(f"SIR_type (type gate):  {sir_metrics['sir_type']}")
print(f"SIR_traj (range gate): {sir_metrics['sir_traj']}")
print(f"Range-gate blocked:    {sir_metrics['n_range_blocked']}")
print(f"Type-gate blocked:     {sir_metrics['n_type_blocked']}")
print(f"Gold paths:            {sir_metrics['gold_paths']}")
print(f"Type gate FNR:         {sir_metrics['fnr_type']}")
print(f"Range gate FNR:        {sir_metrics['fnr_range']}")
print(f"Time:                  {sir_metrics['elapsed_s']}s")
print(f"{'=' * 60}\n")

# Save to Drive.
sir_path = os.path.join(OUTPUT_DIR, "sir_fnr_metrics.json")
with open(sir_path, "w") as f:
    json.dump(sir_metrics, f, indent=2)
print(f"Saved to {sir_path}")

---
## 5. Load LLM

Loads Qwen2.5-1.5B-Instruct. Memory usage: ~3GB (fp16). Fits on any Colab GPU.

In [ ]:
#@title 5. Load LLM

import argparse

LLM = get_registed_model(MODEL_PATH)
parser = argparse.ArgumentParser()
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_PATH
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print(f"Loading {MODEL_PATH}...")
t0 = time.time()
model = LLM(args)
model.prepare_for_inference()

# Suppress sampling warnings for beam search.
model.generation_cfg.temperature = None
model.generation_cfg.top_p = None
model.model.generation_config.temperature = None
model.model.generation_config.top_p = None

load_time = time.time() - t0
n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
vram_used = (
    torch.cuda.memory_allocated(0) / 1e9
    if torch.cuda.is_available()
    else 0
)

print(f"Loaded in {load_time:.1f}s")
print(f"Parameters: {n_params:.1f}M")
print(f"VRAM used:  {vram_used:.1f} GB")
print(f"Tokenizer vocab: {len(model.tokenizer)} tokens")
print("Ready.")

---
## 6. Helper Functions

In [ ]:
#@title 6. Helper Functions

def build_path_trie(tokenizer, path_strings):
    """Build a MarisaTrie from a list of path strings."""
    if not path_strings:
        return None
    tokenized = tokenizer(
        path_strings, padding=False, add_special_tokens=False
    ).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    return MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)


def get_all_paths_and_filter(question_dict, index_len, oracle=None):
    """Enumerate all DFS paths from question entities.

    If oracle is provided, filter through range_gate + type_gate.
    Returns (all_paths, filtered_paths, all_str, filtered_str).
    """
    g = utils.build_graph(question_dict["graph"], undirected=False)
    entities = question_dict.get("q_entity", [])
    if not entities:
        return [], [], [], []

    all_paths = utils.dfs(g, entities, index_len)
    all_str = [utils.path_to_string(p) for p in all_paths]

    if oracle is None:
        return all_paths, all_paths, all_str, all_str

    ans_types = oracle.infer_answer_types(question_dict["question"])
    filtered = []
    for p in all_paths:
        admit = True
        for _, rel, tail in p:
            if not oracle.range_gate(rel, tail):
                admit = False
                break
        if admit:
            terminal = p[-1][2]
            if not oracle.type_gate(terminal, ans_types, len(p), index_len):
                admit = False
        if admit:
            filtered.append(p)

    filtered_str = [utils.path_to_string(p) for p in filtered]
    return all_paths, filtered, all_str, filtered_str


def ask_model(tokenizer, model_obj, question, path_context,
              max_new_tokens=256):
    """Generate an answer given a question and path context."""
    prompt = (
        "Given the question and possible reasoning paths from a "
        "knowledge graph, determine the correct answer. Base your "
        "answer ONLY on the provided paths.\n\n"
        f"Question: {question}\n\n"
        f"Reasoning paths:\n{path_context}\n\n"
        "Answer:"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    )
    inputs = {
        k: v.to(model_obj.model.device) for k, v in inputs.items()
    }

    with torch.no_grad():
        output = model_obj.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return response.strip()


def check_hit(prediction, ground_truth):
    """Check if any ground truth answer appears in the prediction."""
    pred_lower = prediction.lower()
    return any(a.lower() in pred_lower for a in ground_truth)


print("Helpers defined.")

---
## 7. Run with TypeOracle-Filtered Paths

For each question:
1. Enumerate all KG paths via DFS
2. Apply TypeOracle gates (range_gate + type_gate) to filter
3. Build trie from filtered paths only
4. Generate answer using the filtered path set

In [ ]:
#@title 7. Constrained Decoding (TypeOracle-Filtered)

filtered_path = os.path.join(OUTPUT_DIR, "predictions_filtered.jsonl")

# Load existing results for resumption.
processed_ids = set()
if os.path.exists(filtered_path) and not FORCE_RERUN:
    with open(filtered_path) as f:
        for line in f:
            try:
                processed_ids.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass
    print(f"Resuming: {len(processed_ids)} already processed")

fout = open(filtered_path, "a" if processed_ids else "w")

hits_filtered = 0
n_processed = 0
n_total = len(processed_ids)

print(f"\n{'=' * 60}")
print(f"Phase 2: TypeOracle-Filtered Decoding -- {len(dataset)} questions")
print(f"{'=' * 60}\n")

t0 = time.time()

for i, d in enumerate(dataset):
    qid = d["id"]
    if qid in processed_ids:
        continue

    oracle = TypeOracle.from_graph(d["graph"])
    _, filtered, _, filtered_str = get_all_paths_and_filter(
        d, INDEX_LEN, oracle
    )

    if not filtered_str:
        # No paths after filtering -- skip.
        result = {
            "id": qid,
            "question": d["question"],
            "prediction": "",
            "ground_truth": d["answer"],
            "n_paths_all": 0,
            "n_paths_filtered": 0,
            "hit": False,
            "mode": "filtered",
        }
        fout.write(json.dumps(result) + "\n")
        fout.flush()
        n_total += 1
        continue

    # Build context from filtered paths.
    max_paths = 20
    path_lines = []
    for j, p in enumerate(filtered_str[:max_paths]):
        path_lines.append(f"  {j + 1}. {p}")
    ctx = "\n".join(path_lines)

    # Generate answer.
    try:
        pred = ask_model(
            model.tokenizer, model, d["question"], ctx, MAX_NEW_TOKENS
        )
    except Exception as e:
        print(f"  [{i}] Error: {e}")
        pred = ""

    hit = check_hit(pred, d["answer"])
    hits_filtered += int(hit)
    n_total += 1

    # Count blocked paths.
    all_paths_raw = (
        utils.dfs(
            utils.build_graph(d["graph"]),
            d.get("q_entity", []),
            INDEX_LEN,
        )
        if d.get("q_entity")
        else []
    )

    result = {
        "id": qid,
        "question": d["question"],
        "prediction": pred,
        "ground_truth": d["answer"],
        "n_paths_all": len(all_paths_raw),
        "n_paths_filtered": len(filtered),
        "hit": hit,
        "mode": "filtered",
    }
    fout.write(json.dumps(result) + "\n")
    fout.flush()

    # Progress.
    if n_total % 10 == 0:
        elapsed = time.time() - t0
        rate = n_total / elapsed if elapsed > 0 else 0
        print(
            f"  [{n_total}/{len(dataset)}] "
            f"Hit@1: {hits_filtered}/{n_total} "
            f"({hits_filtered / n_total * 100:.1f}%) "
            f"| {rate:.1f} q/s | {elapsed:.0f}s"
        )

fout.close()
elapsed_filtered = time.time() - t0

print(
    f"\nFiltered Hit@1: {hits_filtered}/{n_total} = "
    f"{hits_filtered / max(1, n_total) * 100:.1f}%"
)
print(f"Time: {elapsed_filtered:.1f}s")

---
## 8. Run with Unfiltered Paths (Baseline)

Same as above but uses ALL DFS paths (no TypeOracle filtering).
This is the comparison baseline.

In [ ]:
#@title 8. Unfiltered Decoding (Baseline)

unfiltered_path = os.path.join(OUTPUT_DIR, "predictions_unfiltered.jsonl")

# Load existing results for resumption.
processed_ids_unfilt = set()
if os.path.exists(unfiltered_path) and not FORCE_RERUN:
    with open(unfiltered_path) as f:
        for line in f:
            try:
                processed_ids_unfilt.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass
    print(f"Resuming: {len(processed_ids_unfilt)} already processed")

fout_un = open(unfiltered_path, "a" if processed_ids_unfilt else "w")

hits_unfiltered = 0
n_total_un = len(processed_ids_unfilt)

print(f"\n{'=' * 60}")
print(f"Phase 3: Unfiltered Decoding (Baseline) -- {len(dataset)} questions")
print(f"{'=' * 60}\n")

t0 = time.time()

for i, d in enumerate(dataset):
    qid = d["id"]
    if qid in processed_ids_unfilt:
        continue

    all_paths, _, all_str, _ = get_all_paths_and_filter(
        d, INDEX_LEN, oracle=None
    )

    if not all_str:
        result = {
            "id": qid,
            "question": d["question"],
            "prediction": "",
            "ground_truth": d["answer"],
            "n_paths_all": 0,
            "hit": False,
            "mode": "unfiltered",
        }
        fout_un.write(json.dumps(result) + "\n")
        fout_un.flush()
        n_total_un += 1
        continue

    max_paths = 20
    path_lines = []
    for j, p in enumerate(all_str[:max_paths]):
        path_lines.append(f"  {j + 1}. {p}")
    ctx = "\n".join(path_lines)

    try:
        pred = ask_model(
            model.tokenizer, model, d["question"], ctx, MAX_NEW_TOKENS
        )
    except Exception as e:
        print(f"  [{i}] Error: {e}")
        pred = ""

    hit = check_hit(pred, d["answer"])
    hits_unfiltered += int(hit)
    n_total_un += 1

    result = {
        "id": qid,
        "question": d["question"],
        "prediction": pred,
        "ground_truth": d["answer"],
        "n_paths_all": len(all_paths),
        "hit": hit,
        "mode": "unfiltered",
    }
    fout_un.write(json.dumps(result) + "\n")
    fout_un.flush()

    if n_total_un % 10 == 0:
        elapsed = time.time() - t0
        rate = n_total_un / elapsed if elapsed > 0 else 0
        print(
            f"  [{n_total_un}/{len(dataset)}] "
            f"Hit@1: {hits_unfiltered}/{n_total_un} "
            f"({hits_unfiltered / n_total_un * 100:.1f}%) "
            f"| {rate:.1f} q/s | {elapsed:.0f}s"
        )

fout_un.close()
elapsed_unfiltered = time.time() - t0

print(
    f"\nUnfiltered Hit@1: {hits_unfiltered}/{n_total_un} = "
    f"{hits_unfiltered / max(1, n_total_un) * 100:.1f}%"
)
print(f"Time: {elapsed_unfiltered:.1f}s")

---
## 9. Evaluate & Compare

In [ ]:
#@title 9. Evaluate & Compare

import numpy as np


def load_predictions(path):
    """Load predictions from a JSONL file."""
    results = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    results.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return results


preds_f = load_predictions(filtered_path)
preds_u = load_predictions(unfiltered_path)

# Compute metrics.
n_f = len(preds_f)
n_u = len(preds_u)
hits_f = sum(1 for p in preds_f if p.get("hit", False))
hits_u = sum(1 for p in preds_u if p.get("hit", False))

# Path reduction.
total_all = sum(p.get("n_paths_all", 0) for p in preds_f)
total_filtered = sum(p.get("n_paths_filtered", 0) for p in preds_f)
reduction = (1 - total_filtered / max(1, total_all)) * 100

comparison = {
    "filtered": {
        "n_questions": n_f,
        "hits": hits_f,
        "hit_at_1": round(hits_f / max(1, n_f) * 100, 1),
        "avg_paths": round(total_filtered / max(1, n_f), 1),
        "time_s": round(elapsed_filtered, 1),
    },
    "unfiltered": {
        "n_questions": n_u,
        "hits": hits_u,
        "hit_at_1": round(hits_u / max(1, n_u) * 100, 1),
        "avg_paths": round(total_all / max(1, n_u), 1),
        "time_s": round(elapsed_unfiltered, 1),
    },
    "path_reduction_pct": round(reduction, 1),
    "sir_fnr_metrics": sir_metrics,
}

print("=" * 60)
print("COMPARISON: TypeOracle-Filtered vs Unfiltered")
print("=" * 60)
print(
    f"\n{'Metric':<25} {'Filtered':<12} {'Unfiltered':<12}"
)
print("-" * 49)
print(f"{'Questions':<25} {n_f:<12} {n_u:<12}")
print(f"{'Hits@1':<25} {hits_f:<12} {hits_u:<12}")
print(
    f"{'Hit@1 (%)':<25} "
    f"{comparison['filtered']['hit_at_1']:<12} "
    f"{comparison['unfiltered']['hit_at_1']:<12}"
)
print(
    f"{'Avg paths/question':<25} "
    f"{comparison['filtered']['avg_paths']:<12} "
    f"{comparison['unfiltered']['avg_paths']:<12}"
)
print(
    f"{'Time (s)':<25} "
    f"{comparison['filtered']['time_s']:<12} "
    f"{comparison['unfiltered']['time_s']:<12}"
)
print(f"\nPath reduction: {reduction:.1f}%")

# Per-question comparison.
print("\n--- Per-Question Results (first 20) ---")
print(f"{'Q#':<4} {'Hit(F)':<8} {'Hit(U)':<8} {'Paths F/U':<14} Question")
print("-" * 80)

for i, (pf, pu) in enumerate(zip(preds_f[:20], preds_u[:20])):
    q_short = pf["question"][:40]
    hf = "Y" if pf.get("hit") else "N"
    hu = "Y" if pu.get("hit") else "N"
    paths_f = pf.get("n_paths_filtered", "?")
    paths_u = pf.get("n_paths_all", "?")
    print(f"{i:<4} {hf:<8} {hu:<8} {paths_f}/{paths_u:<10} {q_short}")

print(f"{'=' * 60}")

---
## 10. Save Final Results

In [ ]:
#@title 10. Save Final Results to Google Drive

# Save comparison metrics.
comparison_path = os.path.join(OUTPUT_DIR, "comparison.json")
with open(comparison_path, "w") as f:
    json.dump(comparison, f, indent=2)

# Generate markdown report.
report = (
    f"# TypeOracle Colab Validation Report\n\n"
    f"**Timestamp:** {TIMESTAMP}\n"
    f"**Model:** {MODEL_PATH}\n"
    f"**Dataset:** {DATA_PATH}/{DATASET} ({SPLIT})\n"
    f"**Samples:** {MAX_SAMPLES}\n"
    f"**GPU:** {gpu_name}\n\n"
    f"## SIR/FNR Metrics\n\n"
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| SIR (overall) | {sir_metrics['sir']} |\n"
    f"| SIR_type | {sir_metrics['sir_type']} |\n"
    f"| SIR_traj | {sir_metrics['sir_traj']} |\n"
    f"| Type gate FNR | {sir_metrics['fnr_type']} |\n"
    f"| Range gate FNR | {sir_metrics['fnr_range']} |\n"
    f"| Paths raw | {sir_metrics['total_paths_raw']} |\n"
    f"| Paths filtered | {sir_metrics['total_paths_filtered']} |\n\n"
    f"## Decoding Comparison\n\n"
    f"| Metric | Filtered | Unfiltered |\n"
    f"|--------|----------|------------|\n"
    f"| Hit@1 | {comparison['filtered']['hit_at_1']}% ({hits_f}/{n_f}) | "
    f"{comparison['unfiltered']['hit_at_1']}% ({hits_u}/{n_u}) |\n"
    f"| Avg paths/q | {comparison['filtered']['avg_paths']} | "
    f"{comparison['unfiltered']['avg_paths']} |\n"
    f"| Time | {comparison['filtered']['time_s']}s | "
    f"{comparison['unfiltered']['time_s']}s |\n\n"
    f"**Path reduction:** {reduction:.1f}%\n\n"
    f"## Files\n\n"
    f"- `config.json` -- Experiment configuration\n"
    f"- `sir_fnr_metrics.json` -- SIR/FNR evaluation results\n"
    f"- `predictions_filtered.jsonl` -- TypeOracle-filtered predictions\n"
    f"- `predictions_unfiltered.jsonl` -- Baseline (unfiltered) predictions\n"
    f"- `comparison.json` -- Comparison metrics\n"
    f"- `report.md` -- This report\n"
)

report_path = os.path.join(OUTPUT_DIR, "report.md")
with open(report_path, "w") as f:
    f.write(report)

print("=" * 60)
print("ALL RESULTS SAVED")
print("=" * 60)
print(f"\nOutput directory: {OUTPUT_DIR}")
print("\nFiles:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(fpath)
    print(f"  {fname:<35} {size:>8,} bytes")

print("\nTo download as zip, run the next cell.")

In [ ]:
#@title 11. (Optional) Download Results as ZIP

zip_name = f"GCR_Results_{TIMESTAMP}.zip"
!cd "{DRIVE_BASE}" && zip -r "/content/{zip_name}" "{TIMESTAMP}/"

from google.colab import files
files.download(f"/content/{zip_name}")
print(f"Downloaded {zip_name}")